# INFY Custom Data Source (DB-backed)
This notebook loads INFY daily bars from Postgres via a custom QuantConnect `PythonData` source.

In [65]:
# QuantBook Analysis Tool
qb = QuantBook()

# Force reload so notebook always uses latest db_tick_custom_data.py on disk
import importlib
import db_tick_custom_data
importlib.reload(db_tick_custom_data)
print("db_tick_custom_data loaded from:", db_tick_custom_data.__file__)

DbDailyByTradingsymbol = db_tick_custom_data.DbDailyByTradingsymbol

original_log = DbDailyByTradingsymbol._log.__func__
@classmethod
def quiet_cache_hit_log(cls, message: str) -> None:
    if str(message).startswith("Using existing cache for "):
        return
    return original_log(cls, message)

DbDailyByTradingsymbol._log = quiet_cache_hit_log

db_tick_custom_data loaded from: /Lean/Launcher/bin/Debug/Notebooks/db_tick_custom_data.py


In [64]:
# Request INFY data through custom data subscription
qb.set_start_date(2026, 4, 22)
infy_custom = qb.add_data(DbDailyByTradingsymbol, "INFY", Resolution.DAILY)

# Get 360 daily bars, similar to Cell 2 style
history = qb.history(infy_custom.symbol, 360, Resolution.DAILY)
history.tail()

[DbDailyByTradingsymbol] Refreshing cache for INFY
[DbDailyByTradingsymbol] Resolving instrument for tradingsymbol=INFY
[DbDailyByTradingsymbol] Resolved instrument_token=408065
[DbDailyByTradingsymbol] Fetched 5745 rows from daily_candles, date range [2003-01-01 -> 2026-04-22]
[DbDailyByTradingsymbol] Latest row close/volume = 1257.9/10635696 on 2026-04-22
[DbDailyByTradingsymbol] Wrote cache file: /tmp/lean-db-daily/INFY.csv


close    high     low    open  \
symbol                      time                                         
INFY.DbDailyByTradingsymbol 2026-04-16  1319.2  1331.0  1309.0  1322.1   
                            2026-04-17  1318.7  1328.3  1306.1  1313.0   
                            2026-04-20  1312.6  1324.9  1308.1  1320.0   
                            2026-04-21  1313.2  1325.0  1299.3  1310.0   
                            2026-04-22  1257.9  1297.7  1257.0  1295.0   

                                         value      volume  
symbol                      time                            
INFY.DbDailyByTradingsymbol 2026-04-16  1319.2  18527029.0  
                            2026-04-17  1318.7  12064048.0  
                            2026-04-20  1312.6   7172300.0  
                            2026-04-21  1313.2  10057440.0  
                            2026-04-22  1257.9  10635696.0

## Universe from sector-tree + index-constituents

This section builds a custom universe by intersecting:
- Sector subtree for `IT Services` (from `sector-tree`)
- Constituents of `BSE 1000` (from `index-constituents`)

The output universe is a list of NSE codes.

In [118]:
from typing import List
import importlib

from db_tick_custom_data import SessionLocal
import db_universe_custom_data
importlib.reload(db_universe_custom_data)

payload = db_universe_custom_data.build_universe_payload(
    session_local=SessionLocal,
)

object_store_key = db_universe_custom_data.write_universe_to_object_store(
    qb=qb,
    payload=payload,
    key='portfolio-targets.csv',
)

db_universe_custom_data.ComponentUniverseData.OBJECT_STORE_KEY = object_store_key

# ── Selector filters ───────────────────────────────────────────────────────────
# Mirrors the INDEX_FILTERS / SECTOR_FILTERS constants in main.py.
selector_index_filters = {'BSE 1000'}
selector_sector_filters = {'Information Technology'}
selector_min_weight = 0.0

# Toggle: False → OR logic (odd day / wider)   True → AND logic (even day / narrower)
selector_use_and = False

weight_by_symbol = {}
selected_metadata_by_symbol = {}


def selector_function(alt_coarse: List[PythonData]) -> List[Symbol]:
    """
    Mimics _selector() in main.py.
      selector_use_and = False → OR  (wider  — index OR sector)
      selector_use_and = True  → AND (narrower — index AND sector)
    """
    global weight_by_symbol, selected_metadata_by_symbol

    chosen = {}
    meta = {}
    for d in alt_coarse:
        w = float(d['weight'])
        if w <= selector_min_weight:
            continue

        indexes = set(__import__('json').loads(d['indexes_json'] or '[]'))
        sectors = set(__import__('json').loads(d['sectors_json'] or '[]'))

        matches_index = not selector_index_filters or bool(indexes & selector_index_filters)
        matches_sector = not selector_sector_filters or bool(sectors & selector_sector_filters)

        passes = (matches_index and matches_sector) if selector_use_and else (matches_index or matches_sector)

        if passes:
            chosen[d.symbol] = w
            meta[d.symbol] = {'indexes': sorted(indexes), 'sectors': sorted(sectors)}

    weight_by_symbol = chosen
    selected_metadata_by_symbol = meta
    return list(weight_by_symbol.keys())


custom_universe = qb.add_universe(
    db_universe_custom_data.ComponentUniverseData,
    'ComponentUniverse',
    Resolution.DAILY,
    selector_function,
)

print('Payload symbols  :', payload['total_symbols'])
print('ObjectStore key  :', object_store_key)
print('Index filters    :', selector_index_filters)
print('Sector filters   :', selector_sector_filters)
print('Logic mode       :', 'AND (even day / narrow)' if selector_use_and else 'OR  (odd day / wide)')
print('Universe added   :', custom_universe is not None)

Payload symbols  : 9
ObjectStore key  : portfolio-targets.csv
Index filters    : {'BSE 1000'}
Sector filters   : {'Information Technology'}
Logic mode       : OR  (odd day / wide)
Universe added   : True


In [119]:
import importlib
import db_universe_custom_data
importlib.reload(db_universe_custom_data)

# ── OR mode (odd-day behaviour) ───────────────────────────────────────────────
selector_use_and = False
print('=== OR mode (odd day) — index OR sector must match ===')
selected_or = db_universe_custom_data.run_selector(
    qb=qb,
    key=object_store_key,
    selector_fn=selector_function,
)
or_symbols = [s.value for s in selected_or]
or_weights  = {s.value: weight_by_symbol.get(s) for s in selected_or}
print('Symbols :', or_symbols)
print('Weights :', or_weights)

# ── AND mode (even-day behaviour) ─────────────────────────────────────────────
selector_use_and = True
print('\n=== AND mode (even day) — index AND sector must both match ===')
selected_and = db_universe_custom_data.run_selector(
    qb=qb,
    key=object_store_key,
    selector_fn=selector_function,
)
and_symbols = [s.value for s in selected_and]
and_weights = {s.value: weight_by_symbol.get(s) for s in selected_and}
print('Symbols :', and_symbols)
print('Weights :', and_weights)

# ── Summary ───────────────────────────────────────────────────────────────────
print(f'\nOR  universe : {len(or_symbols)} symbol(s)  → {or_symbols}')
print(f'AND universe : {len(and_symbols)} symbol(s) → {and_symbols}')
only_in_or = sorted(set(or_symbols) - set(and_symbols))
print(f'Dropped on even days (OR only): {only_in_or}')

# ── History for first AND-selected symbol ─────────────────────────────────────
if selected_and:
    sample_symbol = selected_and[0]
    sample_sec = qb.add_data(DbDailyByTradingsymbol, sample_symbol.value, Resolution.DAILY)
    sample_history = qb.history(sample_sec.symbol, 60, Resolution.DAILY)
    print(f'\nHistory for {sample_symbol.value}:')
    display(sample_history.tail())

=== OR mode (odd day) — index OR sector must match ===
[run_selector] Driving selector for date=2026-04-23, rows=9
[run_selector] Selector returned 3 symbol(s): ['INFY', 'KSOLVES', 'SANOFICONR']
Symbols : ['INFY', 'KSOLVES', 'SANOFICONR']
Weights : {'INFY': 0.11111111, 'KSOLVES': 0.11111111, 'SANOFICONR': 0.11111111}

=== AND mode (even day) — index AND sector must both match ===
[run_selector] Driving selector for date=2026-04-23, rows=9
[run_selector] Selector returned 1 symbol(s): ['INFY']
Symbols : ['INFY']
Weights : {'INFY': 0.11111111}

OR  universe : 3 symbol(s)  → ['INFY', 'KSOLVES', 'SANOFICONR']
AND universe : 1 symbol(s) → ['INFY']
Dropped on even days (OR only): ['KSOLVES', 'SANOFICONR']

History for INFY:


close    high     low    open  \
symbol                      time                                         
INFY.DbDailyByTradingsymbol 2026-04-16  1319.2  1331.0  1309.0  1322.1   
                            2026-04-17  1318.7  1328.3  1306.1  1313.0   
                            2026-04-20  1312.6  1324.9  1308.1  1320.0   
                            2026-04-21  1313.2  1325.0  1299.3  1310.0   
                            2026-04-22  1257.9  1297.7  1257.0  1295.0   

                                         value      volume  
symbol                      time                            
INFY.DbDailyByTradingsymbol 2026-04-16  1319.2  18527029.0  
                            2026-04-17  1318.7  12064048.0  
                            2026-04-20  1312.6   7172300.0  
                            2026-04-21  1313.2  10057440.0  
                            2026-04-22  1257.9  10635696.0